# Definining Cohort

**The goal of this notebook is to identify patients with advanced non-small cell lung cancer who received first-line treatment with pembrolizumab or platinum-based chemotherapy. Those who received targeted therapy in later lines are removed.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Platinum-based chemotherapy

In [3]:
therapy = pd.read_csv('../../aNSCLC/data/LineOfTherapy.csv')

In [4]:
(
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .LineName.value_counts()
    .head(20)
)

LineName
Carboplatin,Paclitaxel                                12602
Carboplatin,Pembrolizumab,Pemetrexed                  10174
Carboplatin,Pemetrexed                                 7573
Pembrolizumab                                          7207
Osimertinib                                            3952
Erlotinib                                              3305
Bevacizumab,Carboplatin,Pemetrexed                     3232
Clinical Study Drug                                    3008
Carboplatin,Paclitaxel,Pembrolizumab                   2466
Nivolumab                                              2303
Carboplatin,Paclitaxel Protein-Bound                   2192
Bevacizumab,Carboplatin,Paclitaxel                     1465
Carboplatin,Gemcitabine                                1385
Carboplatin,Paclitaxel Protein-Bound,Pembrolizumab     1304
Pemetrexed                                             1240
Cisplatin,Pemetrexed                                    963
Cisplatin,Etoposide            

In [5]:
therapy_1l = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
)

In [6]:
targeted_agents = [
    # EGFR
    'Osimertinib', 
    'Erlotinib', 
    'Gefitinib', 
    'Afatinib',
    'Amivantamab', 

    # ALK
    'Dacomitinib',
    'Crizotinib', 
    'Ceritinib', 
    'Alectinib', 
    'Brigatinib', 
    'Lorlatinib',

    # ROS1
    'Repotrectinib', 
    'Taletrectinib', 
    'Entrectinib', 
    'Crizotinib',

    # RET
    'Selpercatinib', 
    'Pralsetinib', 

    # BRAF
    'Dabrafenib',
    'Trametinib', 
    'Encorafenib',
    'Binimetinib',

    # NTRK
    'Larotrectinib',
    'Repotrectinib',

    # NRG1
    'Zenocutuzumab',

    # RAS
    'Sotorasib',
    'Adagrasib'
]

pattern_targeted = "|".join(targeted_agents)

In [7]:
ici_agents = [
    'Pembrolizumab', 
    'Nivolumab', 
    'Cemiplimab', 
    'Atezolizumab',
    'Durvalumab', 
    'Avelumab',
    'Ipilimumab',
    'Relatlimab'
]

pattern_ici = "|".join(ici_agents)

In [8]:
(
    therapy_1l[
    therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains(pattern_ici)
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')
    & ~therapy_1l['LineName'].str.contains(pattern_targeted)]
    .LineName.value_counts().head(10)
)

LineName
Carboplatin,Paclitaxel                  12602
Carboplatin,Pemetrexed                   7573
Bevacizumab,Carboplatin,Pemetrexed       3232
Carboplatin,Paclitaxel Protein-Bound     2192
Bevacizumab,Carboplatin,Paclitaxel       1465
Carboplatin,Gemcitabine                  1385
Cisplatin,Pemetrexed                      963
Cisplatin,Etoposide                       946
Carboplatin,Docetaxel                     826
Carboplatin,Etoposide                     410
Name: count, dtype: int64

In [9]:
plat_df = (
    therapy_1l[
    therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains(pattern_ici)
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')
    & ~therapy_1l['LineName'].str.contains(pattern_targeted)]
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'platinum'))

In [10]:
plat_df.sample(3)

,PatientID,LineName,StartDate
44547,FC435482460B6,platinum,2019-11-07
45682,F288F8889BB5F,platinum,2013-04-18
144544,F1542DAC8097D,platinum,2017-04-04


In [11]:
row_ID(plat_df)

(33890, 33890)

In [12]:
maintenance_df = therapy.query('LineNumber == 1').query('IsMaintenanceTherapy == True')

In [13]:
ici_maintenance_ids = (
    maintenance_df[maintenance_df['LineName'].str.contains(pattern_ici)].PatientID
)

In [14]:
plat_df = plat_df[~plat_df.PatientID.isin(ici_maintenance_ids)]

In [15]:
plat_df.shape

(30044, 3)

## 2. Pembrolizumab 

In [16]:
pembro_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Pembrolizumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro'))

In [17]:
pembro_df.sample(3)

,PatientID,LineName,StartDate
146413,F11A894BCB3A1,pembro,2022-12-29
5016,FAC49D19E0C93,pembro,2019-10-21
122200,FA9EFC3A54513,pembro,2021-12-03


In [18]:
row_ID(pembro_df)

(7207, 7207)

## 3. IDs for patients with targeted agents

In [19]:
targeted_ids = (
    therapy
    .query('LineName.str.contains(@pattern_targeted, case=False, na=False)', engine="python")
    .PatientID
)

## 4. Combine dataframes and export to csv 

In [20]:
firstline_pembro_chemo_index = pd.concat([plat_df, pembro_df], axis = 0)

In [21]:
row_ID(firstline_pembro_chemo_index)

(37251, 37251)

In [22]:
# Remove patients any future targeted agent recepit 
firstline_pembro_chemo_index = firstline_pembro_chemo_index[~firstline_pembro_chemo_index.PatientID.isin(targeted_ids)]

In [23]:
row_ID(firstline_pembro_chemo_index)

(34491, 34491)

In [24]:
firstline_pembro_chemo_index.sample(3)

,PatientID,LineName,StartDate
162313,F3CA2B19EA6E7,platinum,2019-07-18
74581,F5EFCB47B78AC,platinum,2014-06-03
20347,F3C6DCC5AAE77,pembro,2019-03-21


In [25]:
firstline_pembro_chemo_index.LineName.value_counts()

LineName
platinum    27487
pembro       7004
Name: count, dtype: int64

In [26]:
firstline_pembro_chemo_index.to_csv('../outputs/pembro_chemo_index.csv', index = False)